# Tensor Basics — Practice Notebook

Practice with PyTorch tensor fundamentals: shape, dtype, device, broadcasting, indexing/slicing, memory layout (view/reshape/permute/contiguous), and a manual implementation of a linear layer.

In [222]:
import numpy as np
import torch

## Task 1. Tensor Inspector: a tensor analyzer

Write a function `inspect_tensor(x, name)` that takes any PyTorch tensor and prints the key information about it.

**The function should show:**
- the tensor's name
- shape
- number of dimensions
- dtype
- device
- number of elements
- size of a single element in bytes
- total memory size of the tensor
- minimum value
- maximum value
- mean value
- standard deviation
- the first 10 elements in flattened form

In [223]:
def inspect_tensor(x: torch.Tensor, name: str = "tensor") -> None:
    """Prints a summary of a tensor's metadata and basic statistics.

    Args:
        x: The tensor to inspect.
        name: A label used to identify the tensor in the printed output.
    """
    print("Name:", name)
    print(" shape:", x.shape)
    print(" dtype:", x.dtype)
    print(" device:", x.device)
    print(" number of elements:", x.numel())
    print(f" element size: {x.element_size()} bytes")
    print(f" total size: {x.numel() * x.element_size()} bytes")

    if x.is_floating_point() or x.dtype in [
        torch.int8,
        torch.int16,
        torch.int32,
        torch.int64,
    ]:
        print(" min:", x.min().item())
        print(" max:", x.max().item())
        print(" mean:", x.float().mean().item())
        print(" std:", x.float().std().item())

    print(" first 10 elements:", x.flatten()[:10])

In [224]:
a = torch.tensor([1, 2, 3, 4, 5])
b = torch.zeros(3, 4)
c = torch.ones(2, 3, 4)
d = torch.randn(32, 10)
e = torch.arange(0, 100, 5)
f = torch.linspace(0, 1, 11)

for i in [a, b, c, d, e, f]:
    inspect_tensor(i)
    print()

Name: tensor
 shape: torch.Size([5])
 dtype: torch.int64
 device: cpu
 number of elements: 5
 element size: 8 bytes
 total size: 40 bytes
 min: 1
 max: 5
 mean: 3.0
 std: 1.5811388492584229
 first 10 elements: tensor([1, 2, 3, 4, 5])

Name: tensor
 shape: torch.Size([3, 4])
 dtype: torch.float32
 device: cpu
 number of elements: 12
 element size: 4 bytes
 total size: 48 bytes
 min: 0.0
 max: 0.0
 mean: 0.0
 std: 0.0
 first 10 elements: tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0.])

Name: tensor
 shape: torch.Size([2, 3, 4])
 dtype: torch.float32
 device: cpu
 number of elements: 24
 element size: 4 bytes
 total size: 96 bytes
 min: 1.0
 max: 1.0
 mean: 1.0
 std: 0.0
 first 10 elements: tensor([1., 1., 1., 1., 1., 1., 1., 1., 1., 1.])

Name: tensor
 shape: torch.Size([32, 10])
 dtype: torch.float32
 device: cpu
 number of elements: 320
 element size: 4 bytes
 total size: 1280 bytes
 min: -2.483513116836548
 max: 3.0195748805999756
 mean: -0.03414551913738251
 std: 1.028820514678955
 

## Task 2. Normalizing batch data via broadcasting

There is a batch of tabular data of size `128 samples × 5 features`.

Each feature needs to be normalized separately so that after normalization each feature has:
- a mean of approximately 0
- a standard deviation of approximately 1

**Do the following:**
- create a tensor with synthetic data
- compute the mean of each feature
- compute the standard deviation of each feature
- normalize the data
- verify the mean and standard deviation after normalization

**Do it in two variants:**
- without `keepdim=True`
- with `keepdim=True`

Compare the shapes of the resulting tensors.

Additionally, write a function `normalize_features(X)` that works for any tensor of shape `batch_size × num_features`.

In [225]:
samples = 128
features = 5
a = 20
b = 13

X = torch.randn(samples, features) * a + b

In [226]:
mean = X.mean(dim=0)
std = X.std(dim=0)

mean_dim = X.mean(dim=0, keepdim=True)
std_dim = X.std(dim=0, keepdim=True)

print(mean, mean.shape)
print(std, std.shape)
print("\nKeep Dim:")
print(mean_dim, mean_dim.shape)
print(std_dim, std_dim.shape)

tensor([15.1597,  9.5301, 12.3718, 15.1441, 13.2716]) torch.Size([5])
tensor([21.4809, 21.1123, 19.5409, 18.4780, 20.9523]) torch.Size([5])

Keep Dim:
tensor([[15.1597,  9.5301, 12.3718, 15.1441, 13.2716]]) torch.Size([1, 5])
tensor([[21.4809, 21.1123, 19.5409, 18.4780, 20.9523]]) torch.Size([1, 5])


In [227]:
def normalize_features(X: torch.Tensor) -> torch.Tensor:
    """Normalizes each feature column to zero mean and unit standard deviation.

    Args:
        X: A tensor of shape ``batch_size x num_features``.

    Returns:
        The normalized tensor, with the same shape as ``X``.
    """
    mean = X.mean(dim=0)
    std = X.std(dim=0)

    X_norm = (X - mean) / std

    return X_norm

In [228]:
X_norm = normalize_features(X)

print(X_norm.mean(dim=0))
print(X_norm.std(dim=0))

tensor([-2.2352e-08,  0.0000e+00, -7.4506e-09, -2.2352e-08, -3.7253e-09])
tensor([1., 1., 1., 1., 1.])


## Task 3. Image Tensor Lab: working with images as tensors

Create a synthetic batch of images of shape `16 × 3 × 32 × 32`, where:
- `16` — number of images in the batch
- `3` — RGB color channels
- `32` — height
- `32` — width

**Do the following:**
- print the batch size
- print the number of channels
- print the height and width
- find the minimum and maximum pixel value
- compute the mean and standard deviation over the whole batch

**Then, using indexing and slicing, get:**
- the first image
- the red channel of the first image
- a center crop of size `16 × 16` for all images
- the top-left corner of size `8 × 8` for all images
- the first 4 images from the batch
- only the green channel for all images

Then, for each RGB channel separately, compute:
- the mean value
- the standard deviation

Normalize the images per channel and verify that after normalization each channel has approximately `mean ≈ 0` and `std ≈ 1`.

Also convert the first image from the PyTorch format `C × H × W` to a display-friendly format: `H × W × C`.

In [229]:
images = torch.randint(0, 256, (16, 3, 32, 32), dtype=torch.float32)

In [230]:
print("batch size:", images.shape[0])
print("number of channels:", images.shape[1])
print("height:", images.shape[2])
print("width:", images.shape[3])

print()
print("min pixel value:", images.min().item())
print("max pixel value:", images.max().item())
print("mean pixel value:", images.mean().item())
print("std pixel value:", images.std().item())

batch size: 16
number of channels: 3
height: 32
width: 32

min pixel value: 0.0
max pixel value: 255.0
mean pixel value: 127.92806243896484
std pixel value: 73.5849838256836


In [231]:
first_image = images[0].clone()
first_red_channel = first_image[0].clone()
center_crop = images[:, :, 8:24, 8:24].clone()
top_left_corner = images[:, :, :8, :8].clone()
first_4_images = images[:4].clone()
green_channel = images[:, 1, :, :].clone()

In [232]:
for channel_n in range(3):
    channel = images[:, channel_n, :, :].float()

    print("channel №", channel_n)
    print("mean:", channel.mean().item())
    print("std:", channel.std().item())
    print()

channel № 0
mean: 128.3419189453125
std: 73.26215362548828

channel № 1
mean: 127.87762451171875
std: 73.64985656738281

channel № 2
mean: 127.56463623046875
std: 73.84416961669922



In [233]:
for channel_n in range(3):
    channel = images[:, channel_n, :, :]

    mean = channel.mean()
    std = channel.std()

    images[:, channel_n, :, :] = (channel - mean) / std

print(images[0])

tensor([[[ 0.9918, -1.5744, -1.6563,  ..., -1.5061, -1.4106,  1.4695],
         [ 0.0363, -0.3459, -0.9874,  ..., -1.3150,  1.3330,  0.6369],
         [ 0.6369, -1.7382,  0.2956,  ...,  0.5550, -0.7008,  1.7288],
         ...,
         [-0.7827, -0.3459, -0.3869,  ..., -1.2604, -0.1685, -1.7518],
         [-0.5370,  0.2137, -1.3287,  ..., -1.0966,  1.4422,  1.5787],
         [-1.6563,  1.3330, -0.0729,  ...,  0.2410,  1.7288, -0.1139]],

        [[ 1.0200, -0.4328, -0.7723,  ..., -1.4647, -0.5143, -0.1748],
         [-1.2746, -1.2339,  1.1286,  ...,  0.1918,  0.9114, -1.6277],
         [-0.6636,  0.6398,  0.1239,  ..., -1.0981, -1.1932,  1.5359],
         ...,
         [ 0.2868,  1.1015, -0.1205,  ..., -1.5326, -0.6365, -1.3290],
         [-0.2699,  0.2868, -0.1341,  ..., -0.6636,  1.1693,  0.8842],
         [ 0.4090,  0.6262,  1.1422,  ...,  0.9114, -1.1253, -1.5734]],

        [[-0.4681,  0.6288, -1.1993,  ..., -0.9691,  1.5361,  0.3851],
         [ 0.5611, -0.9556,  0.2090,  ...,  0

In [234]:
for channel_n in range(3):
    channel = images[:, channel_n, :, :].float()

    print("channel №", channel_n)
    print("mean:", channel.mean().item())
    print("std:", channel.std().item())
    print()

channel № 0
mean: 8.149072527885437e-10
std: 0.9999999403953552

channel № 1
mean: 5.413312464952469e-09
std: 1.0

channel № 2
mean: 1.280568540096283e-09
std: 1.0



In [235]:
converted_image = images[0].permute(1, 2, 0)

print(images[0].shape, "->", converted_image.shape)

torch.Size([3, 32, 32]) -> torch.Size([32, 32, 3])


## Task 4. Manual forward pass of a linear layer

There is a batch of input data of shape `32 × 10`, where:
- `32` — batch size
- `10` — number of features

You need to manually implement the forward pass of a linear layer that transforms the data into shape `32 × 3`.

**Do the following:**
- create an input tensor `X`
- create a weight matrix `W`
- create a bias `b`
- compute the output `Y` via matrix multiplication plus bias
- check the shape of every tensor

**Then compare your manual implementation with `torch.nn.Linear`. You need to:**
- create a `torch.nn.Linear` with `input_dim=10` and `output_dim=3`
- inspect the shape of the layer's `weight` and `bias`
- manually reproduce the forward pass of this layer
- compare the result of the manual forward pass with the built-in layer's output

In [236]:
X = torch.randn(32, 10)
W = torch.randn(10, 3)
b = torch.randn(3)

Y = X @ W + b

print("X:", X.shape)
print("W:", W.shape)
print("b:", b.shape)
print("Y:", Y.shape)

X: torch.Size([32, 10])
W: torch.Size([10, 3])
b: torch.Size([3])
Y: torch.Size([32, 3])


In [237]:
linear = torch.nn.Linear(10, 3)

print("W:", linear.weight.shape)
print("b:", linear.bias.shape)

W: torch.Size([3, 10])
b: torch.Size([3])


In [238]:
Y_builtin = linear(X)
Y_manual = X @ linear.weight.T + linear.bias

torch.allclose(Y_builtin, Y_manual)

True

## Task 5. Device, NumPy interop, and contiguous memory

Run an experiment with devices and memory.

### Part 1. Device

Determine the available device: `cuda` if a GPU is available, otherwise `cpu`.

Create a tensor on the CPU and move it to the chosen device.

In [239]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("awailable device:", device)

a = torch.randn(10)
a_device = a.to(device)

print("original device:", a.device)
print("switched device:", a_device.device)

awailable device: cpu
original device: cpu
switched device: cpu


### Part 2. NumPy interop

Create a NumPy array and convert it to a PyTorch tensor.

In [240]:
np_array = np.arange(5, dtype=np.float32)
torch_array = torch.from_numpy(np_array)

print("numpy:", np_array)
print("torch:", torch_array)

numpy: [0. 1. 2. 3. 4.]
torch: tensor([0., 1., 2., 3., 4.])


In [241]:
np_array[0] = 3

print("numpy:", np_array)
print("torch:", torch_array)

numpy: [3. 1. 2. 3. 4.]
torch: tensor([3., 1., 2., 3., 4.])


### Part 3. View, reshape, permute, contiguous

Create a 3D tensor and change the order of its axes using `permute`.

In [242]:
x = torch.arange(24).reshape(2, 3, 4)

x_perm = x.permute(1, 2, 0)

print("x shape:", x.shape)
print("x_perm shape:", x_perm.shape)
print("x contiguous:", x.is_contiguous())
print("x_perm contiguous:", x_perm.is_contiguous())

x shape: torch.Size([2, 3, 4])
x_perm shape: torch.Size([3, 4, 2])
x contiguous: True
x_perm contiguous: False


In [243]:
try:
    x_perm.view(2, 12)
except Exception as e:
    print(e)

view size is not compatible with input tensor's size and stride (at least one dimension spans across two contiguous subspaces). Use .reshape(...) instead.


In [244]:
x_perm.contiguous().view(2, 12)

tensor([[ 0, 12,  1, 13,  2, 14,  3, 15,  4, 16,  5, 17],
        [ 6, 18,  7, 19,  8, 20,  9, 21, 10, 22, 11, 23]])